In [1]:
!pip install pandas sentence-transformers openpyxl

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import pandas as pd

# Load Excel file
df = pd.read_csv("/content/drive/My Drive/GSoC/AutoEIT Sample Transcriptions for Scoring(38006-2A).csv",encoding='ISO-8859-1')

# Check column names
print("Columns:", df.columns)

Columns: Index(['Sentence', 'Stimulus', 'Transcription Rater 1', 'Score'], dtype='object')


In [12]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')


# Clean text function
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()

    # Remove noise patterns
    text = re.sub(r"\[.*?\]", "", text)   # remove [gibberish], [pause]
    text = re.sub(r"xxx", "", text)
    text = re.sub(r"\.\.\.", "", text)
    text = re.sub(r"[^a-zA-Záéíóúñü\s]", "", text)

    return text.strip()

# Similarity → score mapping
def get_score(similarity):
    if similarity > 0.85:
        return 4
    elif similarity > 0.70:
        return 3
    elif similarity > 0.50:
        return 2
    elif similarity > 0.30:
        return 1
    else:
        return 0

scores = []

# Loop through rows
for i, row in df.iterrows():
    prompt = clean_text(row['Stimulus'])
    learner = clean_text(row['Transcription Rater 1'])

    if prompt == "" or learner == "":
        scores.append(0)
        continue

    emb1 = model.encode(prompt, convert_to_tensor=True)
    emb2 = model.encode(learner, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()
    score = get_score(similarity)

    scores.append(score)

# Add scores to dataframe
df['Score'] = scores

# Save output file
output_file = "scored_output.xlsx"
df.to_excel(output_file, index=False)

print("✅ Scoring completed! File saved as:", output_file)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Scoring completed! File saved as: scored_output.xlsx


In [13]:
files.download("scored_output.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>